The output of snlc_sim.exe are ".fits" files. This notebook aims to get photometry data from snlc_sim results and merge different simulation results to one csv file.

In [ ]:
import pandas as pd
import numpy as np
import os
from pprint import pprint
from astropy.io import fits
import matplotlib.pyplot as plt
import matplotlib as mpl
import tqdm

SIM_PATH = "<BASE_DIR>/SNANA/SNDATA_ROOT/SIM/"
mpl.rcParams.update({
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'font.size':14
})
TYPE="NSBH_TRAIN"
sim_name = f"LSST_KN_{TYPE}"

In [ ]:
injections = pd.read_csv(f"<BASE_DIR>/ML+GW+KN/dataset/O5_sim_{TYPE.lower()}/injections_final.csv", index_col='simulation_id')
sim_ids = injections.index
print("Total number of injections:", len(sim_ids))

In [ ]:
sim_dirs = os.listdir(os.path.join(SIM_PATH,sim_name))
covered_sim_ids = [int(d.split('_')[-1]) for d in sim_dirs]
print("Number of injections covered in SNANA SIM output:", len(covered_sim_ids))

In [ ]:
fail_ids = np.loadtxt(f"<BASE_DIR>/data/{sim_name}/failed_sim_ids.txt", dtype=int)
fail_ids = np.unique(fail_ids)
print("Number of failed snlc_sim:", len(fail_ids))

In [ ]:
for fail_id in fail_ids:
    try:
        covered_sim_ids.pop(covered_sim_ids.index(fail_id))
    except:
        None
print("Number of success snlc_sim:",len(covered_sim_ids))

In [ ]:
np.savetxt(f"<BASE_DIR>/data/{sim_name}/success_sim_ids.txt", covered_sim_ids, fmt='%d')

In [ ]:
covered_sim_ids = sorted(covered_sim_ids)
sample_num = {}
valid_sim_ids = []
NLIBID = {}

for i, sim_id in tqdm.tqdm(enumerate(covered_sim_ids), total=len(covered_sim_ids)):
    dump_file = os.path.join(SIM_PATH, sim_name, f"{sim_name}_{sim_id}", f"{sim_name}_{sim_id}.DUMP")
    readme_file = os.path.join(SIM_PATH, sim_name, f"{sim_name}_{sim_id}", f"{sim_name}_{sim_id}.README")
    with open(dump_file, 'r') as f:
        lines = f.readlines()
        if len(lines)==0:
            continue
        sample_num[sim_id] = len(lines) - 8  # Exclude header line
        if sample_num[sim_id]>0:
            valid_sim_ids.append(sim_id)
        # print(f"Simulation {sim_id} has {sample_num[i]} samples.")
    with open(readme_file, "r") as f:
        line = f.readlines()[18]
        NLIBID[sim_id]=int(line.split()[1])

print("Successfully snlc_sim:", len(sample_num))
print("Total number of light curves in SNANA SIM output:", sum(sample_num.values()))
#print("Valid simulation IDs with LCs:", valid_sim_ids)
#print("NLIBID for all simulations:", NLIBID)
print("Total number of LIBIDs:", sum(NLIBID.values()))
print("Total numbers of valid simulations:", len(valid_sim_ids))

In [ ]:
# calculate fraction
print("Total number of LIBIDs:", sum(NLIBID.values()))
print("Total number of light curves in SNANA SIM output:", sum(sample_num.values()))

frac = sum(sample_num.values())/sum(NLIBID.values())
print("Fraction: Detected lightcurves / Simulated lightcurves:", frac)

In [ ]:
plt.hist(sample_num.values(), bins=50, range=[1,10000])
plt.title("Numbers of detected lightcurves for each events", fontsize=16)
plt.xlabel("Number of detected lightcurves",fontsize=16)
plt.ylabel("Frequency", fontsize=16)
plt.xticks(fontsize=14)
plt.show()

## Analyse distribution of simulations with results

In [ ]:
valid_inj = injections.loc[valid_sim_ids].copy()
valid_inj.reset_index(inplace=True)

valid_inj["NLIBID"] = valid_inj.index.map(NLIBID).astype("Int64")
valid_inj["NDET_LC"] = valid_inj.index.map(sample_num).astype("Int64")

valid_inj.to_csv(f"<BASE_DIR>/data/{sim_name}/gw_catalog.csv", index=False)
valid_inj

In [ ]:
success_inj = injections.loc[sample_num.keys()].copy()
success_inj.reset_index(inplace=True)
success_inj

In [ ]:
valid_dist = valid_inj['distance']
success_dist = success_inj['distance']
plt.figure(figsize=(16,6))
plt.subplot(121)
plt.hist(valid_dist, bins=20,color='b',label=f"Num:{len(valid_dist)}")
plt.xlabel("Distance(Mpc)", fontsize=16)
plt.ylabel("Frequency", fontsize=16)
plt.legend()
plt.title("Luminosity distance distribution of detected events", fontsize=16)
plt.subplot(122)
plt.hist(success_dist,bins=20,color='r',label=f"Num:{len(success_dist)}")
plt.xlabel("Distance(Mpc)")
plt.ylabel("Frequency")
plt.legend()
plt.title("Distance distribution of all events")
plt.figure(figsize=(8,6))
plt.scatter(valid_dist, valid_inj['NDET_LC'])
plt.xlabel("Distance(Mpc)", fontsize=16)
plt.ylabel("NDET_LC", fontsize=16)
plt.title("Number of detected lightcurve fro detected events")

In [ ]:
# get redshift for each lightcurve
redshift_lcs = np.empty(0)
for id in tqdm.tqdm(valid_sim_ids, total=len(valid_sim_ids)):
    head_file = os.path.join(SIM_PATH,sim_name, f"{sim_name}_{id}", f"{sim_name}_{id}_HEAD.FITS")
    try:
        head_hdul = fits.open(head_file)
        head_data = head_hdul[1].data
        redshift_lcs = np.concatenate([redshift_lcs, head_data['REDSHIFT_FINAL']])
    except:
        continue
print("Numbers of light curves:",len(redshift_lcs))
plt.hist(redshift_lcs, range=(0.0,0.22), bins=25)
plt.xlabel("Redshift")
plt.ylabel("Frequency")
plt.title("Redshift distribution of detected light curves")

In [ ]:
valid_mej_dyn = valid_inj['mej_dyn']
valid_mej_wind = valid_inj['mej_wind']
valid_m1 = valid_inj['mass1']
valid_m2 = valid_inj['mass2']

# Heatmaps of ejecta masses vs (mass1, mass2)
bins = 20

# 2D mean for dynamical ejecta
H_sum_dyn, xedges, yedges = np.histogram2d(valid_m1, valid_m2, bins=bins, weights=valid_mej_dyn)
H_count, _, _ = np.histogram2d(valid_m1, valid_m2, bins=[xedges, yedges])
H_mean_dyn = H_sum_dyn / np.where(H_count == 0, np.nan, H_count)

# 2D mean for wind ejecta
H_sum_wind, _, _ = np.histogram2d(valid_m1, valid_m2, bins=[xedges, yedges], weights=valid_mej_wind)
H_mean_wind = H_sum_wind / np.where(H_count == 0, np.nan, H_count)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

im0 = axs[0].pcolormesh(xedges, yedges, H_mean_dyn.T, cmap='viridis', shading='auto')
axs[0].set_xlabel('valid_m1 (M_sun)')
axs[0].set_ylabel('valid_m2 (M_sun)')
axs[0].set_title('Mean Dynamical Ejecta Mass')
c0 = fig.colorbar(im0, ax=axs[0])
c0.set_label('valid_mej_dyn (M_sun)')

im1 = axs[1].pcolormesh(xedges, yedges, H_mean_wind.T, cmap='viridis', shading='auto')
axs[1].set_xlabel('valid_m1 (M_sun)')
axs[1].set_ylabel('valid_m2 (M_sun)')
axs[1].set_title('Mean Wind Ejecta Mass')
c1 = fig.colorbar(im1, ax=axs[1])
c1.set_label('mej_wind (M_sun)')

plt.show()

In [ ]:
plt.scatter(valid_mej_dyn,valid_mej_wind,color='r',alpha=0.5,s=5)
plt.xlabel("Dynamical ejecta mass")
plt.ylabel("Wind ejecta mass")
plt.vlines(x=[0.001,0.02],ymin=0.01,ymax=0.13,colors='b',ls='--')
plt.hlines(y=[0.01,0.13],xmin=0.001,xmax=0.02,colors='b',ls='--')
plt.vlines(x=[0.01,0.09],ymin=0.01,ymax=0.09,colors='k',ls='--')
plt.hlines(y=[0.01,0.09],xmin=0.01,xmax=0.09,colors='k',ls='--')
plt.title("Ejecta mass of Detected simulations")
print("Minimum and Maximum of dynamical ejcta mass:", min(valid_mej_dyn),max(valid_mej_dyn))
print("Minimum and Maximum of wind ejcta mass:", min(valid_mej_wind),max(valid_mej_wind))
print("Model range of dynamical ejcta mass:[0.001,0.02]")
print("Model range of wind ejcta mass:[0.01,0.13]")

In [ ]:
if TYPE=="BNS" or TYPE=="BNS_AUG":
    valid_mej_total = valid_inj['mej_total']
    success_mej_tot = success_inj['mej_total']
elif "NSBH" in TYPE:
    valid_mej_total = valid_inj['mej_tot']
    success_mej_tot = success_inj['mej_tot']
plt.figure(figsize=(16,6))
plt.subplot(1,2,1)
plt.hist(valid_mej_total,bins=20, color='b', alpha=0.75, label=f"Num:{len(valid_mej_total)}")
plt.xlabel("Mass of ejecta(M_sun)")
plt.ylabel("Frequency")
plt.title("Ejecta mass distribution of detected events")
plt.legend()
plt.subplot(1,2,2)
plt.hist(success_mej_tot, color='r', alpha=0.75, bins=20,label=f"Num:{len(success_mej_tot)}")
plt.xlabel("Mass of ejecta(M_sun)")
plt.ylabel("Frequency")
plt.legend()
plt.title("Ejecta mass distribution of all events")

In [ ]:
valid_obs_angle = valid_inj['inclination']
plt.figure(figsize=(16,6))
plt.subplot(121)
plt.hist(valid_obs_angle, bins=25, color='b', label=f'Num:{len(valid_obs_angle)}')
plt.xlabel("Inclination(rad)")
plt.ylabel("Frequency")
plt.title("Observing Angle distribution of detected events")
plt.legend()
success_obs_angle = success_inj['inclination']
plt.subplot(122)
plt.hist(success_obs_angle, bins=25, color='r', label=f"Num:{len(success_obs_angle)}")
plt.xlabel("Inclination(rad)")
plt.ylabel("Frequency")
plt.title("Observing Angle distribution of all events")
plt.legend()

In [ ]:
valid_phi = valid_inj['phi']
plt.figure(figsize=(16,6))
plt.subplot(121)
plt.hist(valid_phi, bins=25, color='b', label=f'Num:{len(valid_phi)}')
plt.xlabel("Phi(deg)")
plt.ylabel("Frequency")
plt.title("Half-Opening Angle distribution of detected events")
plt.legend()
success_phi = success_inj['phi']
plt.subplot(122)
plt.hist(success_phi, bins=25, color='r', label=f"Num:{len(success_phi)}")
plt.xlabel("Phi(deg)")
plt.ylabel("Frequency")
plt.title("Half-Opening Angle distribution of all events")
plt.legend()

In [ ]:
valid_mjd_time = valid_inj['mjd_time']
initial_mjd_time = success_inj['mjd_time']
plt.figure(figsize=(16,6))
plt.subplot(121)
plt.hist(valid_mjd_time, bins=25, color='b')
plt.xlabel("Merger time of BNS events(MJD)")
plt.ylabel("Frequency")
plt.title("Merger time distribution of detected events")
plt.subplot(122)
plt.hist(initial_mjd_time, bins=25, color='r')
plt.xlabel("Inclination(rad)")
plt.ylabel("Frequency")
plt.title("Merger time distribution of all events")

In [ ]:
import healpy as hp
from healpy.visufunc import projscatter

valid_theta = np.pi/2 - valid_inj['latitude'].values
valid_phi = valid_inj['longitude'].values

initial_theta = np.pi/2 - success_inj['latitude'].values
initial_phi = success_inj['longitude'].values

hp.mollview()
hp.graticule()
projscatter(valid_theta, valid_phi, c='b', marker='o', s=5, label='Detected events')
#projscatter(initial_theta, initial_phi, c='r')

## Analyse parameters of failed ids


In [ ]:
fail_inj = injections.loc[fail_ids].copy()
fail_inj

In [ ]:
fail_mej_dyn = fail_inj['mej_dyn']
fail_mej_wind = fail_inj['mej_wind']
fail_m1 = fail_inj['mass1']
fail_m2 = fail_inj['mass2']

# Heatmaps of ejecta masses vs (mass1, mass2)
bins = 20

# 2D mean for dynamical ejecta
H_sum_dyn, xedges, yedges = np.histogram2d(fail_m1, fail_m2, bins=bins, weights=fail_mej_dyn)
H_count, _, _ = np.histogram2d(fail_m1, fail_m2, bins=[xedges, yedges])
H_mean_dyn = H_sum_dyn / np.where(H_count == 0, np.nan, H_count)

# 2D mean for wind ejecta
H_sum_wind, _, _ = np.histogram2d(fail_m1, fail_m2, bins=[xedges, yedges], weights=fail_mej_wind)
H_mean_wind = H_sum_wind / np.where(H_count == 0, np.nan, H_count)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

im0 = axs[0].pcolormesh(xedges, yedges, H_mean_dyn.T, cmap='viridis', shading='auto')
axs[0].set_xlabel('fail_m1 (M_sun)')
axs[0].set_ylabel('fail_m2 (M_sun)')
axs[0].set_title('Mean Dynamical Ejecta Mass')
c0 = fig.colorbar(im0, ax=axs[0])
c0.set_label('fail_mej_dyn (M_sun)')

im1 = axs[1].pcolormesh(xedges, yedges, H_mean_wind.T, cmap='viridis', shading='auto')
axs[1].set_xlabel('fail_m1 (M_sun)')
axs[1].set_ylabel('fail_m2 (M_sun)')
axs[1].set_title('Mean Wind Ejecta Mass')
c1 = fig.colorbar(im1, ax=axs[1])
c1.set_label('mej_wind (M_sun)')

plt.show()

In [ ]:
success_mej_dyn = success_inj['mej_dyn']
success_mej_wind = success_inj['mej_wind']
plt.scatter(fail_mej_dyn, fail_mej_wind,color='b',label='Failure',alpha=0.5,s=10, linewidths=0)
plt.scatter(success_mej_dyn,success_mej_wind,color='r',label='Success',alpha=0.5,s=5, linewidths=0)
plt.xlabel("Dynamical ejecta mass")
plt.ylabel("Wind ejecta mass")
plt.legend()
plt.vlines(x=[0.001,0.02],ymin=0.01,ymax=0.13,colors='r',ls='--')
plt.hlines(y=[0.01,0.13],xmin=0.001,xmax=0.02,colors='r',ls='--')
plt.vlines(x=[0.01,0.09],ymin=0.01,ymax=0.09,colors='k',ls='--')
plt.hlines(y=[0.01,0.09],xmin=0.01,xmax=0.09,colors='k',ls='--')
plt.title("Ejecta mass of failed simulations")
print("Minimum and Maximum of dynamical ejcta mass:", min(fail_mej_dyn),max(fail_mej_dyn))
print("Minimum and Maximum of wind ejcta mass:", min(fail_mej_wind),max(fail_mej_wind))
print("Model range of dynamical ejcta mass:[0.001,0.02]")
print("Model range of wind ejcta mass:[0.01,0.13]")

In [ ]:
plt.scatter(fail_inj['mass1_source'], fail_inj['mass2_source'], color='b', label='Failure', alpha=0.5, s=5, linewidths=0)
plt.scatter(success_inj['mass1_source'], success_inj['mass2_source'], color='r', label='Success', alpha=0.5, s=5, linewidths=0)
plt.xlabel("Mass1_source(M_sun)")
plt.ylabel("Mass2_source(M_sun)")
plt.legend()
plt.title("Mass1 vs Mass2 of detected and failed events")

In [ ]:
#plt.scatter(valid_inj['mass1_source'], valid_inj['spin1z'], color='b', label='Detected', alpha=0.5, s=5, linewidths=0)
plt.scatter(success_inj['mass1_source'], success_inj['spin1z'], color='r', label='Success', alpha=0.5, s=5, linewidths=0)
plt.scatter(fail_inj['mass1_source'], fail_inj['spin1z'], color='k', label='Failure', alpha=0.5, s=5, linewidths=0)
plt.xlabel("Mass1_source(M_sun)")
plt.ylabel("Spin1z")
plt.legend()
plt.title("Mass1 vs Spin1z of detected and failed events")

In [ ]:
fail_phi = fail_inj['phi']
fail_costheta = fail_inj['costheta']
success_phi = success_inj['phi']
success_costheta = success_inj['costheta']
plt.scatter(fail_phi, fail_costheta,color='b',label='Failure',alpha=0.5,s=5)
plt.scatter(success_phi,success_costheta,color='r',label='Success',alpha=0.5,s=5)
plt.xlabel("Half-Opening angle(Phi)")
plt.ylabel("Cos(Theta)")
plt.legend()
plt.xlim((0,90))
plt.ylim((0,1))
plt.title("Angle parameters of failed simulations")
print("Minimum and Maximum of Half-Opening angle(Phi):", min(fail_phi),max(fail_phi))
print("Minimum and Maximum of Cos(Theta):", min(fail_costheta),max(fail_costheta))
print("Model range of Half-Opening angle(Phi):[1,90]")
print("Model range of Cos(Theta):[0,1]")